### 16 - MLflow Tracking

##### 1. Notebook Objective


This notebook formalizes MLflow experiment tracking for the Product Defect
Computer Vision project.

The models have already been trained and persisted in MLflow. Therefore,
this notebook does not retrain them.

The goals are to understand:

1. Why experiment tracking is needed.
2. The relationship between MLflow experiments and runs.
3. What should be recorded as parameters, metrics, tags, and artifacts.
4. How trained PyTorch models are persisted in MLflow.
5. What model signatures and input examples represent.
6. How MLflow supports reproducibility, traceability, comparison, and
   Responsible AI.
7. How the persisted Custom CNN and ResNet18 runs connect to model
   registration and deployment.

##### 2. Why do we need experiment tracking?

Imagine we train a model today and get: Accuracy = 99.58%

Six weeks later someone asks: Which model produced that result?

We might remember: ResNet18

But then come more questions:

- Which learning rate?
- How many epochs?
- Which preprocessing?
- Was the backbone frozen?
- Was layer4 fine-tuned?
- Which validation loss selected the model?
- Which dataset?
- Which code version?
- Where are the trained weights?
- Which metrics belonged to this model?

Without tracking, we could end up with:

- model_final.pth
- model_final_v2.pth
- model_really_final.pth
- model_final_best.pth

That isn't a production model-management strategy.

Instead:

``` text

Training execution
        ↓
MLflow Run
        ↓
everything associated with
that execution

```

##### 3. Experiment versus Run

An **experiment** is a container for related runs.

For example:

``` text

Product Defect Computer Vision
│
├── Custom CNN run
│
├── ResNet feature-extraction run
│
├── ResNet fine-tuning run
│
└── Future experiments...
```

A **run** represents one tracked execution.

Conceptually:

``` text

Experiment
│
├── Run A
│   ├── parameters
│   ├── metrics
│   ├── tags
│   ├── artifacts
│   └── model
│
└── Run B
    ├── parameters
    ├── metrics
    ├── tags
    ├── artifacts
    └── model

Our project already has real examples:

- Custom CNN
Run ID:8b154a58aaae400fbb941a7d8299dc19

and:

- Fine-tuned ResNet18
Run ID: 3f9b44ae55a449c5850f0b48c4ee4fa6

The Run ID uniquely identifies the tracked run.

##### 4. What belongs inside an MLflow run?

Think of a run as the model's development record.

``` text

MLflow Run
│
├── Parameters
│
├── Metrics
│
├── Tags
│
├── Artifacts
└── Model

```
Let's understand each.

##### 5. Parameters

Parameters describe **how the model was configured.**

For our custom CNN:

- learning_rate = 0.001
- epochs = 5
- batch_size = 32
- image_height = 224
- image_width = 224
- cnn_channels = 32,64,128
- classifier_hidden_dim = 64

For transfer learning:

- architecture = resnet18
- pretrained_weights = DEFAULT
- feature_extraction_lr = 0.001
- fine_tuning_lr = 0.0001
- feature_extraction_epochs = 3
- fine_tuning_epochs = 2

Parameters generally answer: **What configuration did we use?**

##### 6. Metrics

Metrics describe **what happened during training or evaluation**.

Examples:

- train_loss
- validation_loss
- train_accuracy
- validation_accuracy
- test_accuracy
- precision
- recall
- F1

Training metrics can also vary by step/epoch:

``` text

Epoch 1
val_loss = ...

Epoch 2
val_loss = ...

Epoch 3
val_loss = ...
```

MLflow can preserve this history.

Conceptually:
``` text

Parameter
learning_rate = 0.001
       ↓
configuration


Metric
validation_loss = 0.034
       ↓
observed result
```

A useful rule:

``` text

PARAMETER
"What did I choose?"

METRIC
"What did I measure?"

```

##### 7. Tags

Tags provide descriptive metadata.

For example:

- model_type = resnet18
- task = image_classification
- project = product_defect_computer_vision
- training_strategy = partial_fine_tuning

Tags are particularly useful for searching and organizing runs.

Think:

- Parameter → configuration
- Metric    → measured number
- Tag       → description/context

##### 8. Artifacts

Artifacts are files produced by the run.

Examples:

- model files
- plots
- confusion matrices
- evaluation reports
- JSON metadata

And importantly: MLflow model is stored as an artifact associated with the run.

This explains something you noticed earlier.

Inside the MLflow model package we saw something similar to:

``` text

model/
└── data/
    └── model.pth

```
You did not manually create: /Volumes/.../model.pth

MLflow created and managed the serialized PyTorch model as part of its model artifact.

##### 9. Why we didn't manually save .pth

Recall the question we had during custom CNN training:

Should we save the PyTorch weights manually to our Unity Catalog Volume?

For this project, our persistence flow is:

``` text

Training
   ↓
Best model selected in memory
   ↓
MLflow logs the model
   ↓
MLflow-managed model artifact
   ↓
Persistent trained model

```

rather than:

``` text

Training
   ↓
torch.save(...)
   ↓
some manually managed model.pth
   ↓
remember where it is
   ↓
manually load it later

```

A raw PyTorch checkpoint might contain:

torch.save(
    model.state_dict(),
    "model.pth",
)

That's perfectly valid PyTorch.

But then we are responsible for managing:

- Which architecture created these weights?
- Which preprocessing was used?
- Which PyTorch version?
- Which run produced them?
- Which metrics belong to them?
- Which model version should be deployed?

MLflow gives us a managed abstraction around that model.

``` text

MLflow Model
│
├── serialized model
├── model flavor
├── environment/dependencies
├── metadata
├── signature          ← when supplied
└── input example      ← when supplied

```
So .pth isn't bad. We simply don't need a second manual persistence mechanism for this project.

##### 10. Logging a PyTorch model

In [0]:
#Conceptually, we previously did something like:

mlflow.pytorch.log_model(
    pytorch_model=model,
    artifact_path="model",
)

#And later loaded it using:

model = mlflow.pytorch.load_model(
    model_uri
)

#For example:

model_uri = (
    "runs:/"
    "3f9b44ae55a449c5850f0b48c4ee4fa6"
    "/model"
)

model = mlflow.pytorch.load_model(
    model_uri
)


This URI is important: runs:/<RUN_ID>/model

Break it apart:

runs:/
   ↓
look inside an MLflow run

3f9b44...
   ↓
specific Run ID

/model
   ↓
model artifact logged in that run

So:

runs:/RUN_ID/model

means:

Load the model artifact named model associated with this MLflow run.

##### 11. But our previous logging generated warnings

During training we saw warnings indicating that the model was logged without:

- signature
- input example

The model still worked.

We proved that because later notebooks successfully did:

- mlflow.pytorch.load_model(...)
- and evaluated the model.

So:

``` text

No signature
No input example
      ↓
does NOT mean
model wasn't saved

But for production-style model packaging, adding them is useful.

```

Let's understand why.

##### 12. What is an MLflow model signature?

**A model signature describes the model's input/output contract.**

Think back to our PyTorch architecture.

Our ResNet expects:

- Input: [B, 3, 224, 224]
- Output: [B, 2]

For example:

``` text

Input

[1, 3, 224, 224]
       ↓
    ResNet18
       ↓
[1, 2]

```
The signature formalizes information about the model's expected input and output schema.

Conceptually:

``` text

              MODEL CONTRACT

INPUT                         OUTPUT
────────                      ──────
image tensor                  logits
batch                         batch
3 channels                    2 classes
224 × 224

```
Why does that matter?

Without an input contract:

``` text

Application
    ↓
sends something
    ↓
Model

```
What if the application accidentally sends:

[1, 224, 224]

instead of:

[1, 3, 224, 224]

Or sends something represented differently from what the serving model expects?

A defined interface makes the model easier to validate and integrate.

##### 13. This is similar to function contracts

Suppose we have:

def predict(
    image_tensor: torch.Tensor,
) -> torch.Tensor:
    ...

We're communicating:

- input  → tensor
- output → tensor

MLflow signatures serve a related purpose at the model boundary.

``` text

Application
     ↓
expected input schema
     ↓
MLflow Model
     ↓
expected output schema

```

This becomes especially important during deployment.

##### 14. What is an input example?

**An input example is a representative example of valid model input stored with the model.**

Conceptually:

``` text

Signature
    ↓
"What is the expected schema?"

Input example
    ↓
"What does a valid input actually look like?"

For a tabular churn model, an input example might resemble:

tenure    MonthlyCharges    Contract
12        72.5              Month-to-month

``` 
For our vision model, the underlying neural network consumes numerical image data after preprocessing.

Conceptually:

``` text

Raw image
   ↓
preprocessing
   ↓
tensor
[3,224,224]
   ↓
batch dimension
   ↓
[1,3,224,224]

```

An appropriate model-level example should represent the form expected at the boundary we choose to expose.

And this introduces an important deployment design question.

##### 15. What exactly should our deployed model accept?

There are two possible architectural boundaries.

**Option A — model accepts preprocessed tensors**

``` text

Client
  ↓
load image
  ↓
resize
  ↓
normalize
  ↓
create tensor
  ↓
endpoint
  ↓
ResNet18

```

This exposes preprocessing responsibility to the client.

**Option B — serving application accepts raw image data**

``` text

Client
   ↓
raw image
   ↓
endpoint
   ↓
inference code
   ├── decode image
   ├── resize
   ├── normalize
   └── tensor
          ↓
       ResNet18
          ↓
       prediction

```
For our eventual application, Option B is usually the more useful external API design.

Why?

We don't want every caller to have to know: 

- ResNet18 normalization mean/std
- resize/crop rules
- RGB conversion
- tensor channel order

Those are implementation details that should ideally live in our reusable inference code.

This is exactly why we created: src/image_preprocessing.py

and will use: src/inference.py

during deployment.

##### 16. Training preprocessing versus inference preprocessing

This is an important connection back to Notebook 03.

Training can use randomness:

TRAIN

- RandomResizedCrop
- RandomHorizontalFlip
- ...

Inference must not.

INFERENCE

- deterministic resize/crop
- expected normalization
- ...

Our pretrained ResNet evaluation transform is based on:

- weights = ResNet18_Weights.DEFAULT
- weights.transforms()

That preprocessing needs to remain consistent when the model is used for inference.

Conceptually:

``` text

TRAINING / EVALUATION

Image
 ↓
ResNet preprocessing
 ↓
Model

```

and:

PRODUCTION

``` text

Image
 ↓
same expected inference preprocessing
 ↓
same model

```
If production accidentally changes preprocessing:

``` text

Development

RGB
224×224
correct normalization
        ↓
      model


Production

RGB
different resize
no normalization
        ↓
      model
      
```
then the model weights may be identical but its predictions can change substantially.

That's why we previously classified training/serving consistency as a Responsible AI reliability concern too.

##### 17. How is a signature created?

MLflow provides:

from mlflow.models import infer_signature

Conceptually:

``` text

signature = infer_signature(
    model_input,
    model_output,
)

```
MLflow looks at:

example input
+
example output

and derives the corresponding model schema.

For example:

example_input = ...

``` text

with torch.no_grad():
    example_output = model(
        example_input
    )

signature = infer_signature(
    example_input.cpu().numpy(),
    example_output.cpu().numpy(),
)

```
Then it can be supplied when logging the model.

##### 18. Production-style model logging pattern

In [0]:
#For future training runs, the pattern becomes approximately:

from mlflow.models import infer_signature

model.eval()

example_images, _ = next(
    iter(data.val_loader)
)

example_input = (
    example_images[:1]
    .to(device)
)

with torch.no_grad():

    example_output = model(
        example_input
    )

signature = infer_signature(
    example_input
    .detach()
    .cpu()
    .numpy(),

    example_output
    .detach()
    .cpu()
    .numpy(),
)

In [0]:
mlflow.pytorch.log_model(
    pytorch_model=model,
    name="model",
    signature=signature,
    input_example=(
        example_input
        .detach()
        .cpu()
        .numpy()
    ),
)

##### 19. Better multi-cell Databricks pattern

In [0]:
if mlflow.active_run() is not None:
    mlflow.end_run()

run = mlflow.start_run(
    run_name="resnet18_transfer_learning"
)

run_id = run.info.run_id

print(
    "Run ID:",
    run_id,
)

Then subsequent cells can log to the active run.
At the end: mlflow.end_run()

``` text

Cell 1
start run
   │
   ↓
Cell 2
log parameters
   │
   ↓
Cell 3
training
   │
   ↓
Cell 4
log metrics
   │
   ↓
Cell 5
log model
   │
   ↓
Cell 6
end run

```

##### 20. Parameters we should track for this project

In [0]:
#A production-style ResNet run could include:

mlflow.log_params(
    {
        "architecture": "resnet18",
        "num_classes": data.num_classes,
        "image_height": 224,
        "image_width": 224,
        "batch_size": 32,
        "feature_extraction_epochs": 3,
        "fine_tuning_epochs": 2,
        "feature_extraction_lr": 1e-3,
        "fine_tuning_lr": 1e-4,
        "fine_tuned_layer": "layer4",
        "pretrained_weights": "DEFAULT",
    }
)

#These tell us how the model was produced.

##### 21. Metrics we should track

We have multiple categories.

**Training metrics**

- train_loss
- train_accuracy
- validation_loss
- validation_accuracy

potentially logged per epoch.

**Model-selection metrics**

- best_validation_loss
- best_validation_accuracy
- best_epoch

**Final evaluation metrics**

After model selection is complete:

- test_accuracy
- defect_precision
- defect_recall
- defect_f1
- ok_precision
- ok_recall
- ok_f1

**RAI-relevant evaluation metrics**

From our later assessment:

- missed_defects
- missed_defect_rate
- false_defect_alarms
- exact_development_test_duplicates
- non_duplicate_test_accuracy

There is a subtle point here:

Not every metric has to be generated during the training notebook.

Evaluation and RAI notebooks can generate additional evidence associated with the model lifecycle.

##### 22. Metrics versus artifacts

Suppose we have:

test_accuracy = 0.9958

That's naturally a metric.

But what about:

``` text

[[450, 3],
 [0, 262]]

```

We could log four numbers.

But a confusion-matrix visualization is often more useful as an artifact:

artifacts/ confusion_matrix.png

Likewise:

- Grad-CAM comparisons
- RAI report
- class-distribution plot
- training curves

can be artifacts.

A helpful rule:

``` text

Small scalar value
      ↓
Metric

Configuration value
      ↓
Parameter

Descriptive metadata
      ↓
Tag

File / plot / report
      ↓
Artifact

```

##### 23. MLflow and reproducibility

Now we can see the larger purpose.

Without MLflow: "I trained a ResNet and got 99.5%."

With tracking:

``` text

Run ID
   ↓
architecture
   ↓
hyperparameters
   ↓
training metrics
   ↓
validation metrics
   ↓
model artifact
   ↓
environment
   ↓
signature
   ↓
input example

```
This doesn't automatically guarantee perfect reproducibility—data versions, random operations, infrastructure and external dependencies still matter—but it gives us much stronger traceability.

##### 24. MLflow and Responsible AI

MLflow is primarily an MLOps/model-lifecycle tool, but it also supports important RAI practices.

Consider accountability.

Someone asks:

Which model produced this prediction?

We can potentially trace:

``` text

Endpoint
   ↓
Registered model version
   ↓
MLflow model
   ↓
Training run
   ↓
parameters
metrics
artifacts

```

That's much stronger than: some_model.pth


**⚖️ RAI — Traceability**


Responsible AI includes knowing:

- which model was used
- where it came from
- how it was evaluated
- what limitations were known
- which version was deployed

MLflow helps us maintain that evidence.

##### 25. Experiment Tracking vs Model Registry

This distinction becomes important for Notebook 17.

**MLflow Tracking**

Answers: What happened during model development?

``` text

Experiment
   ↓
Runs
   ↓
params
metrics
artifacts
models

```

**Model Registry**

Answers: Which model versions are managed as deployment candidates?

``` text

Registered Model
       ↓
Version 1
Version 2
Version 3

```
So:

``` text

TRAINING
   ↓
MLflow Run
   ↓
Model Artifact
   ↓
REGISTER
   ↓
Registered Model Version
   ↓
DEPLOY

```
A run and a registered model version are **not the same thing**.

##### 26. Let's inspect our existing runs

In [0]:
import mlflow

CUSTOM_CNN_RUN_ID =  "8b154a58aaae400fbb941a7d8299dc19"
RESNET18_RUN_ID =    "3f9b44ae55a449c5850f0b48c4ee4fa6"

In [0]:
#Retrieve the custom CNN run:

cnn_run = mlflow.get_run(
    CUSTOM_CNN_RUN_ID
)

print(
    "CNN Run ID:",
    cnn_run.info.run_id,
)

print(
    "Status:",
    cnn_run.info.status,
)

print(
    "\nParameters:"
)

for key, value in (
    cnn_run.data.params.items()
):
    print(
        f"{key}: {value}"
    )

print(
    "\nMetrics:"
)

for key, value in (
    cnn_run.data.metrics.items()
):
    print(
        f"{key}: {value}"
    )

In [0]:
#ResNet:

resnet_run = mlflow.get_run(
    RESNET18_RUN_ID
)

print(
    "ResNet Run ID:",
    resnet_run.info.run_id,
)

print(
    "Status:",
    resnet_run.info.status,
)

print(
    "\nParameters:"
)

for key, value in (
    resnet_run.data.params.items()
):
    print(
        f"{key}: {value}"
    )

print(
    "\nMetrics:"
)

for key, value in (
    resnet_run.data.metrics.items()
):
    print(
        f"{key}: {value}"
    )

##### Interpretation of Existing MLflow Runs

Both training runs completed successfully and contain the primary
configuration and model-selection metrics required to understand how the
models were trained.

The Custom CNN run records its architecture type, optimizer, loss function,
learning rate, batch size, epoch count, and best validation results.

The ResNet18 run provides additional transfer-learning traceability. It
records the pretrained architecture, feature-extraction stage, partial
fine-tuning of `layer4`, learning rates for both stages, validation-loss
model-selection criterion, and the selected fine-tuning stage.

The ResNet18 validation results improved from approximately:

- 95.18% validation accuracy after feature extraction
- to 99.50% after partial fine-tuning.

The 99.50% value is a validation result used during model development and
should not be confused with final test performance.

Later evaluation measured approximately:

- 99.58% accuracy on the original 715-image test split;
- 99.54% accuracy on the 651-image non-duplicate test subset.

The existing runs therefore provide strong training and model-selection
traceability. Future production-style runs can improve this further by
including model signatures, input examples, explicit preprocessing
metadata, and final evaluation artifacts.

##### 27. Verify the model artifacts

In [0]:
CUSTOM_CNN_MODEL_URI = (
    f"runs:/{CUSTOM_CNN_RUN_ID}/model"
)

RESNET18_MODEL_URI = (
    f"runs:/{RESNET18_RUN_ID}/model"
)

print(
    "Custom CNN:",
    CUSTOM_CNN_MODEL_URI,
)

print(
    "ResNet18:",
    RESNET18_MODEL_URI,
)

These URIs provide the bridge from: Experiment tracking to: Model registration

which is exactly what Notebook 17 will use.

##### 28. final MLflow mental model

``` text

                 MLFLOW

              EXPERIMENT
                  │
        ┌─────────┴─────────┐
        ↓                   ↓
      RUN A               RUN B
   Custom CNN             ResNet18
        │                   │
        ├─ Parameters       ├─ Parameters
        ├─ Metrics          ├─ Metrics
        ├─ Tags             ├─ Tags
        ├─ Artifacts        ├─ Artifacts
        └─ Model            └─ Model
                                 │
                                 ↓
                         Model Signature
                         Input Example
                                 │
                                 ↓
                         MODEL REGISTRY
                                 │
                                 ↓
                           Model Version
                                 │
                                 ↓
                            Deployment

```


``` text

MLflow Run
   ↓
"Here is what happened during
this model-development execution."

MLflow Model Artifact
   ↓
"Here is the trained model
produced by that run."

Registered Model Version
   ↓
"Here is a governed version
of a model we want to manage."

Serving Endpoint
   ↓
"Here is a running service
that applications can call."

```

##### Key Learnings

1. MLflow Tracking records the history and outputs of model-development
   executions.

2. An MLflow experiment groups related runs.

3. An MLflow run represents one tracked execution and has a unique Run ID.

4. Parameters describe model and training configuration.

5. Metrics represent measured training, validation, or evaluation results.

6. Tags provide descriptive metadata that helps organize and identify runs.

7. Artifacts store files and outputs such as models, plots, and reports.

8. MLflow can persist a complete PyTorch model without requiring a separate
   manually managed `.pth` file in project storage.

9. A model URI such as `runs:/<run_id>/model` identifies a model artifact
   associated with a specific MLflow run.

10. A model signature describes the expected model input/output interface.

11. An input example provides a representative valid model input.

12. Model signatures and input examples improve model documentation and
    production integration.

13. Training preprocessing and production inference preprocessing must remain
    consistent.

14. Multi-cell Databricks notebooks require deliberate MLflow run lifecycle
    management so later cells continue logging to the intended run.

15. Model-development metrics and later evaluation/RAI evidence are different
    parts of the same model lifecycle.

16. MLflow improves reproducibility and traceability, but reproducibility also
    depends on data, code, random state, environment, and infrastructure.

17. MLflow traceability supports Responsible AI accountability by connecting
    model behavior to identifiable model versions and development evidence.

18. MLflow Tracking and the Model Registry serve different purposes:
    Tracking records development history, while the Registry manages model
    versions intended for downstream use.

##### Conclusion

This notebook formalized MLflow experiment tracking for the Product Defect
Computer Vision project without retraining the existing models.

The Custom CNN and fine-tuned ResNet18 are already persisted as MLflow model
artifacts associated with identifiable training runs. These runs provide a
traceable connection between model development and the trained model
artifacts used in later evaluation, Explainable AI, and Responsible AI
analysis.

MLflow experiments organize related development runs, while individual runs
record parameters, metrics, tags, artifacts, and model outputs.

The notebook also introduced model signatures and input examples. Although
the existing training runs were logged without these components, they remain
valid persisted models and have been successfully reused throughout the
project. Future production-style model logging should include a clearly
defined model interface, signature, and representative input example where
appropriate.

MLflow tracking also contributes to model governance and Responsible AI by
providing traceability between a trained model and the evidence associated
with its development and evaluation.

The next step is to move from a model artifact associated with a development
run to a governed model version in the model registry.